# Wealth & Brokerage — Statistical Testing Workbook

**Hands-on notebook** using the included fictional CSVs. Run from the `notebooks/` directory. Start with the observations and plots, then change inputs. See `README.md` for assumptions, detailed test catalog, and interpretation cautions.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
clients = pd.read_csv(ROOT/'data/clients.csv')
orders = pd.read_csv(ROOT/'data/orders.csv')
strategies = pd.read_csv(ROOT/'data/strategies.csv')
market = pd.read_csv(ROOT/'data/market_daily.csv')
print({k: len(v) for k,v in [('clients',clients),('orders',orders),('strategies',strategies),('market',market)]})
display(clients.head())

## How to interpret
For every category: (1) Identify grain, (2) state H0/H1, (3) validate assumptions, (4) visualize, (5) calculate p-value and effect size, (6) interpret evidence *without* claiming causation.

**Tip:** `run_tests.py` is the complete executable reference for all nine families, with 33 output rows. In notebook cells below, call each category and inspect plots in `outputs/`.

## Means

**Decision question:** Compare strategies and same-client latency before/after.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['means'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Variance

**Decision question:** Examine portfolio strategy variance and volatility.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['variance'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Proportions

**Decision question:** Compare conversion rates; inspect asset-class mix.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['proportions'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Association

**Decision question:** Test risk profile vs advisory adoption, and pilot complaint counts.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['association'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Correlation

**Decision question:** Assess digital score vs trading frequency.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['correlation'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Regression

**Decision question:** Assess revenue drivers conditional on segment.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['regression'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Nonparametric

**Decision question:** Consider highly skewed execution latency.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['nonparametric'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Distribution

**Decision question:** Check whether daily market returns are normally distributed.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['distribution'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Timeseries

**Decision question:** Compare level and return stationarity and serial correlations.

**Your task:** Write H0 and H1 before running. Afterward, find the appropriate generated PNG and write a business conclusion plus one caveat.

In [ ]:
import run_tests
run_tests.records.clear()
run_tests.FAMILIES['timeseries'](alpha=0.05)
display(pd.DataFrame(run_tests.records))
# Generated figures are available in ../outputs (open in notebook or file browser).

## Fully worked example: a two-proportion z-test

**H0:** onboarding rate is identical for Control and New App. **H1:** rates differ. Treatment arms were randomly assigned in the synthetic dataset. Each client contributes one binary outcome. Business effect = absolute conversion difference, not p-value.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
g=clients.groupby('experiment_arm')['onboarded'].agg(['sum','count'])
t, c = g.loc['New App'], g.loc['Control']
p_t=t['sum']/t['count']; p_c=c['sum']/c['count']
stat,p=proportions_ztest([t['sum'],c['sum']],[t['count'],c['count']])
ci=confint_proportions_2indep(t['sum'],t['count'],c['sum'],c['count'],compare='diff',method='newcomb')
print('New App:',round(p_t,4),'Control:',round(p_c,4))
print('Absolute lift (percentage points):',round(100*(p_t-p_c),2))
print('95% CI for lift (percentage points):',tuple(round(100*x,2) for x in ci))
print('z-statistic:',round(stat,3),'p-value:',round(p,5))
g.assign(rate=g['sum']/g['count'])['rate'].plot.bar(title='Onboarding: Control vs New App',ylabel='Rate',rot=0); plt.show()

## Independent practice
1. Does digital engagement differ by risk tier? (Which test? Why?)
2. Are complaint rates different between experiment arms? Calculate rate difference and CI.
3. Would comparing orders as independent observations bias standard errors if a client places multiple orders?
4. Does a small p-value necessarily imply that a strategy outperforms *after trading fees and risk*?
5. Replace synthetic daily returns with historical Apple returns from the URL in `README.md` (network required) and rerun distribution/time-series tests.